# Lưu ROI126 và bank đã sửa lemma thành một Dataset

**CPU/Accelerator None, Internet ON**, không cần Add Input.
Bật secret KAGGLE_API_TOKEN; nếu tên secret là codex, sửa SECRET_NAME ở cell đầu.
OWNER phải có quyền xem Output126 và tạo Dataset; không dán token vào code/chat.

Tải đúng8file gốc từ version126, kiểm fullSHA cả regions.npz. Thêm bank đã sửa
và correctionreceipt từ Gitcommit cố định, kiểmSHA và tái tạo bảng theo delta.
Giữ nguyên mọi file gốc; file để mã hóa CLIP text là **bank_words_corrected.json**.
Không chạy lại POS/ROI/model và không tạo embeddings ở đây.

Tạo **Private Dataset mới**, không ghi đè dataset cũ. Nếu slug đã có, kiểm tra
trangDataset trước khi thử lại. GiữOutput126 đến khi xử lýDataset hoàn tất.


In [ ]:
from pathlib import Path
KERNEL_VERSION='ducanh2403/2-vi-cai-kia-qua-nhieu/126'
OWNER='ducanh2403'
SLUG='csa-real-v126-lemma-fixed'
TITLE='CSA real v126 lemma fixed assets'
SECRET_NAME='KAGGLE_API_TOKEN'
DESTINATION=Path('/kaggle/working/csa_real_corrected_dataset')


In [ ]:
import os,re,subprocess,sys
from kaggle_secrets import UserSecretsClient
if KERNEL_VERSION!='ducanh2403/2-vi-cai-kia-qua-nhieu/126':raise ValueError('This notebook pins126; do not substitute latest or another version.')
if not re.fullmatch(r'[a-zA-Z0-9_-]+',OWNER) or not re.fullmatch(r'[a-z0-9]+(?:-[a-z0-9]+)*',SLUG):raise ValueError('Invalid OWNER/SLUG')
if not 6<=len(SLUG)<=50 or not 6<=len(TITLE)<=50:raise ValueError('TITLE/SLUG must have6–50characters')
try:
    token=UserSecretsClient().get_secret(SECRET_NAME).strip()
    if not token:raise ValueError('empty')
except Exception:raise RuntimeError('Enable the configured SECRET_NAME in Add-ons > Secrets. Do not paste token in code/chat.') from None
def run_cli(args,authenticated=True):
    environment=os.environ.copy()
    for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):environment.pop(key,None)
    if authenticated:environment['KAGGLE_API_TOKEN']=token
    environment['PYTHONUNBUFFERED']='1'
    with subprocess.Popen(args,env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace') as process:
        for line in process.stdout:print(line.replace(token,'[REDACTED]'),end='',flush=True)
        code=process.wait()
    if code:raise RuntimeError(f'Command failed ({code}); see redacted output. No automatic fallback or dataset overwrite.')

run_cli([sys.executable,'-m','pip','install','--upgrade','--force-reinstall','--quiet','https://github.com/Kaggle/kaggle-cli/archive/6894864ff4095cf98e9e36d3a694ff5fc9ba3818.zip'],authenticated=False)
VERSION_PREFLIGHT="import tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\n\n# Exercise the installed download implementation without contacting Kaggle.\napi = KaggleApi()\nwith tempfile.TemporaryDirectory() as directory, patch.object(api, 'build_kaggle_client') as factory:\n    endpoint = factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n    endpoint.return_value = SimpleNamespace(files=[], next_page_token=None, log=None)\n    api.kernels_output('ducanh2403/2-vi-cai-kia-qua-nhieu/126', path=directory)\n    request = endpoint.call_args.args[0]\n    assert getattr(request, 'version_label', None) == 'v126', 'Installed CLI ignores the requested version; STOP before download.'\n    assert request.user_name == 'ducanh2403'\n    assert request.kernel_slug == '2-vi-cai-kia-qua-nhieu'\nprint('Version routing preflight PASS: request.version_label=v126; no network call.', flush=True)\n"
run_cli([sys.executable,'-u','-c',VERSION_PREFLIGHT])


In [ ]:
import hashlib,json
INVENTORY={'kernel_version': 'ducanh2403/2-vi-cai-kia-qua-nhieu/126', 'source_review_zip_sha256': '9bd5a8b1e643186fa001a20d2782705898227fac54590b2ca468b8adf53a18b8', 'files': {'vocabulary.json': '924be210aa0daa8b0968a6d69ad2c66b3de603f49c35f4aa67247c75d6173fc7', 'bank_words.json': '450fc969e0842ca3270d1f899b7e8fe594639260644ad5815a69430fe48264b1', 'pilot_metadata.json': '892e90561dc99b7ca0e4e58615a5a750d11e6df0ce84d4c1482a9322a5c00beb', 'regions.npz': '054f79bcf13de02255893d5741e695ebfb453f66bc5e488234fba65702904b6c', 'receipt.json': 'b95196eba5dc19ef534485a0855b3ea194097ade05a1d44095194a6548fa1e42', 'summary.json': '206cfc43053179d65f9696debfd6b549f8f59a556a79b28c1dd69e42db2a8207', 'source_receipt.json': '8cc88416da138a7f2248d6371e6c1f03f67d3329fbf1334bb34289921daceabb', 'software_freeze.txt': '117f207643e1c4e13fea4f596069e16c15dca6927f7eca793a2b8ad38fb09b45'}}
SOURCE_CODE='"""Byte-preserving CPU transport only; no API, NumPy archive or model loading."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport shutil\n\nNAMES = {\'vocabulary.json\', \'bank_words.json\', \'pilot_metadata.json\', \'regions.npz\',\n         \'receipt.json\', \'summary.json\', \'source_receipt.json\', \'software_freeze.txt\'}\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(4 << 20), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef package(source, destination, inventory):\n    expected = inventory[\'files\']\n    if inventory[\'kernel_version\'] != \'ducanh2403/2-vi-cai-kia-qua-nhieu/126\' or set(expected) != NAMES:\n        raise ValueError(\'Transport requires exact version126/eight original files\')\n    if any(not re.fullmatch(\'[0-9a-f]{64}\', h) for h in expected.values()):\n        raise ValueError(\'Invalid pinned digest\')\n    source, destination = Path(source).resolve(), Path(destination).resolve()\n    if source == destination or source in destination.parents or destination in source.parents:\n        raise ValueError(\'Separate non-nested folders required\')\n    for name, digest in expected.items():\n        if (source/name).is_symlink() or not (source/name).is_file() or sha(source/name) != digest:\n            raise ValueError(\'Missing/changed source payload: \'+name)\n    receipt = json.loads((source/\'receipt.json\').read_bytes())\n    summary = json.loads((source/\'summary.json\').read_bytes())\n    if receipt[\'status\'] != \'CPU_PREPARATION_COMPLETE\' or summary[\'status\'] != \'CPU_PREPARATION_COMPLETE\':\n        raise ValueError(\'Source is not completed CPU preparation\')\n    if receipt[\'schema\'] != \'csa_paper_words_roi_v1\' or receipt[\'semantic_embeddings_created\'] or receipt[\'training_ready\']:\n        raise ValueError(\'Unexpected source schema or training state\')\n    if summary[\'regions\'][\'images\'] != 320 or summary[\'regions\'][\'train_images\'] != 256 or summary[\'regions\'][\'val_images\'] != 64:\n        raise ValueError(\'Wrong pilot cohort\')\n    if receipt[\'payload_sha256\'] != {n: expected[n] for n in (\'vocabulary.json\', \'bank_words.json\', \'pilot_metadata.json\', \'regions.npz\')}:\n        raise ValueError(\'Receipt manifest differs\')\n    destination.mkdir(parents=True, exist_ok=True)\n    if any(p.is_symlink() or p.name not in NAMES | {\'transport_receipt.json\'} for p in destination.iterdir()):\n        raise ValueError(\'Preserve unrelated/partial destination; choose a fresh folder\')\n    needed = sum((source/n).stat().st_size for n in expected if not (destination/n).exists())\n    if shutil.disk_usage(destination).free < needed + (128 << 20):\n        raise RuntimeError(\'Insufficient copy space\')\n    for name, digest in expected.items():\n        target = destination/name\n        if target.exists():\n            if not target.is_file() or sha(target) != digest:\n                raise ValueError(\'Changed destination preserved: \'+name)\n            continue\n        partial = destination/(name+\'.partial\')\n        shutil.copyfile(source/name, partial)\n        if sha(partial) != digest:\n            raise ValueError(\'Copy digest differs: \'+name)\n        partial.replace(target)\n    report = dict(status=\'DATASET_FOLDER_READY\', source_kernel_version=inventory[\'kernel_version\'],\n        source_review_zip_sha256=inventory[\'source_review_zip_sha256\'], files=expected,\n        full_original_file_hashes_verified=True, original_bytes_unchanged=True,\n        model_executed=False, semantic_embeddings_created=False, semantic_word_lists_cleared=False,\n        lexical_warning=\'rid18575 vs ride881: suspected lemma problem, CPU token trace required before CLIP text encoding\')\n    (destination/\'transport_receipt.json\').write_text(json.dumps(report, indent=2)+\'\\n\', encoding=\'utf-8\')\n    return report\n'
assert hashlib.sha256(SOURCE_CODE.encode()).hexdigest()=='600cb085ef388d46bcdaf089990759724076d52135e2453c66c3c20c04e896b7'
exec(compile(SOURCE_CODE,'package_real_dataset.py','exec'))


In [ ]:
import tempfile,shutil
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<512*1024**2:
    raise RuntimeError('Need512MiB free for staged download/copy; preserve existing files.')
DOWNLOAD=Path(tempfile.mkdtemp(prefix='csa-real-v126-',dir='/tmp'))
pattern=r'(^|/)(?:'+'|'.join(re.escape(n) for n in INVENTORY['files'])+r')$'
print('Downloading EXACT Output:',KERNEL_VERSION,flush=True)
run_cli([sys.executable,'-m','kaggle','kernels','output',KERNEL_VERSION,'-p',str(DOWNLOAD),'--file-pattern',pattern])
matches=[p.parent for p in DOWNLOAD.rglob('receipt.json') if sha(p)==INVENTORY['files']['receipt.json']]
if len(matches)!=1:raise RuntimeError(f'Output126 returned {len(matches)} matching receipts. STOP, no latest fallback.')
report=package(matches[0],DESTINATION,INVENTORY)
print('Eight original126 files verified/copied unchanged; next add corrected words.',flush=True)


In [ ]:
OVERLAY={'bank_words_corrected.json': {'url': 'https://raw.githubusercontent.com/Supzxjee/Image_Captioning/27d884df3b0621b0c7745ad1e7be316628489da9/experiments/csa_paper/lemma_correction_review_text/bank_words_corrected.json', 'sha256': 'cec76a0229f30361235a4b64e9712ac6bb2e79fc62a9241079a7af3fa95d13ac'}, 'correction_receipt.json': {'url': 'https://raw.githubusercontent.com/Supzxjee/Image_Captioning/27d884df3b0621b0c7745ad1e7be316628489da9/experiments/csa_paper/lemma_correction_review_text/correction_receipt.json', 'sha256': 'aa3f7c738124973715d009c0a75e6ea1cd3d11d4f4a06c1cb22906c9086e2123'}}
RULE_SOURCE='"""Narrow, evidence-backed metadata rule; no model imports."""\nfrom collections import Counter\n\n\ndef corrected_lemma(surface, pos, original_lemma):\n    if pos == \'VERB\' and surface.lower() in (\'riding\', \'rides\') and original_lemma.lower() == \'rid\':\n        return \'ride\'\n    return original_lemma\n\n\ndef corrected_bank(original, moved_count):\n    if not isinstance(moved_count, int) or moved_count <= 0 or original[\'minimum_count\'] != 5:\n        raise ValueError(\'Positive measured count and original min5 policy required\')\n    before = Counter({row[\'word\']: row[\'count\'] for row in original[\'verb\']})\n    if before[\'rid\'] < moved_count or not before[\'ride\']:\n        raise ValueError(\'Correction exceeds baseline counts or baseline ride absent\')\n    after = before.copy()\n    after[\'rid\'] -= moved_count\n    after[\'ride\'] += moved_count\n    if sum(after.values()) != sum(before.values()):\n        raise ValueError(\'Count mass changed\')\n    result = dict(original)\n    result[\'verb\'] = [dict(word=w, count=after[w]) for w in sorted(after, key=lambda w: (-after[w], w)) if after[w] >= 5]\n    result[\'lemma_amendment\'] = dict(version=\'verb_riding_rides_rid_to_ride_v1\',\n        surface_forms=[\'riding\', \'rides\'], pos=\'VERB\', original_lemma=\'rid\', corrected_lemma=\'ride\',\n        moved_tokens=moved_count, other_verbs_unchanged=True, nouns_unchanged=True)\n    return result, dict(before_rid=before[\'rid\'], before_ride=before[\'ride\'],\n        after_rid=after[\'rid\'], after_ride=after[\'ride\'], moved_tokens=moved_count,\n        below_minimum_omitted_mass=sum(n for n in after.values() if n < 5))\n'
assert hashlib.sha256(RULE_SOURCE.encode()).hexdigest()=='538de6980996762de4333f96d1f56fb203318c210f68046e87e5e2e1fbfa8dc1'

exec(compile(RULE_SOURCE,'lemma_correction.py','exec'))
import urllib.request
for name, item in OVERLAY.items():
    if Path(name).name!=name: raise ValueError('Invalid overlay name')
    payload=urllib.request.urlopen(item['url'],timeout=120).read()
    if hashlib.sha256(payload).hexdigest()!=item['sha256']: raise ValueError('Corrected overlay hash differs: '+name)
    target=DESTINATION/name
    if target.is_symlink() or (target.exists() and target.read_bytes()!=payload): raise ValueError('Changed destination preserved: '+name)
    if not target.exists(): target.write_bytes(payload)
receipt=json.loads((DESTINATION/'correction_receipt.json').read_bytes())
old_words=json.loads((DESTINATION/'bank_words.json').read_bytes())
new_words=json.loads((DESTINATION/'bank_words_corrected.json').read_bytes())
if receipt['status']!='BANK_LEMMA_CORRECTION_COMPLETE' or receipt['preserved_roi_source_version']!=126: raise ValueError('Wrong correction receipt')
if receipt['source_bank_sha256']!=INVENTORY['files']['bank_words.json']: raise ValueError('Corrected bank belongs to another source')
if receipt['corrected_bank_sha256']!=OVERLAY['bank_words_corrected.json']['sha256']: raise ValueError('Receipt/new bank SHA differs')
expected_words, delta=corrected_bank(old_words,receipt['delta']['moved_tokens'])
expected_words['lemma_amendment']['source_bank_sha256']=receipt['source_bank_sha256']
if new_words!=expected_words or delta!=receipt['delta']: raise ValueError('Corrected count/noun policy differs')
for name, expected in INVENTORY['files'].items():
    if sha(DESTINATION/name)!=expected: raise ValueError('Original126 file changed: '+name)
merged=dict(status='ORIGINAL126_AND_CORRECTED_WORDS_READY',original_kernel_version=KERNEL_VERSION,
    original_files_sha256=INVENTORY['files'],corrected_overlay_sha256={n:v['sha256'] for n,v in OVERLAY.items()},
    selected_bank_file='bank_words_corrected.json',selected_bank_sha256=receipt['corrected_bank_sha256'],
    original_files_unchanged=True,semantic_embeddings_created=False,model_executed=False,
    note='Raw bank and historical transport warning preserved; selected bank has the measured narrow lemma correction.')
(DESTINATION/'corrected_assets_receipt.json').write_text(json.dumps(merged,indent=2)+chr(10),encoding='utf-8')
print('Verified corrected bank selected; originals unchanged. No POS/ROI/model rerun.',flush=True)


In [ ]:
CREATE_DATASET_SCRIPT="import sys\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi = KaggleApi()\napi.authenticate()\nresult = api.dataset_create_new(sys.argv[1], public=False, quiet=False,\n                                convert_to_csv=False, dir_mode='skip')\nif result is None or str(getattr(result, 'status', '')).lower() != 'ok' or getattr(result, 'error', None):\n    raise RuntimeError('Dataset creation rejected: ' + str(getattr(result, 'error', 'Missing success response')))\nprint('Dataset creation accepted by API:', result.url, flush=True)\n"

metadata_path=DESTINATION/'dataset-metadata.json'
metadata=dict(title=TITLE,id=f'{OWNER}/{SLUG}',licenses=[dict(name='other')],
    description='Private research assets preserved from exact Output126: TRAIN noun/verb POS word lists and320-image pooled raw CLIP regions. Upstream model/data terms retained. Original word list preserved; separate selected bank has measured riding/rides lemma correction; not CLIP text embeddings or a trained caption model. No image pixels included. Original receipt/provenance preserved.')
if metadata_path.exists():raise RuntimeError('Preserve existing metadata; choose a fresh destination.')
try:
    metadata_path.write_text(json.dumps(metadata,indent=2),encoding='utf-8')
    run_cli([sys.executable,'-u','-c',CREATE_DATASET_SCRIPT,str(DESTINATION)])
finally:metadata_path.unlink(missing_ok=True)
print('API accepted creation; processing may still be pending. Do not repeatedly create.',flush=True)
print(f'https://www.kaggle.com/datasets/{OWNER}/{SLUG}',flush=True)
try:run_cli([sys.executable,'-m','kaggle','datasets','status',f'{OWNER}/{SLUG}'])
except RuntimeError:print('Status not confirmed; check Dataset page before retrying.',flush=True)


Gửi path mount thực tế sau khi Dataset xử lý xong và được attach.
Mặc định dự kiến `/kaggle/input/datasets/ducanh2403/csa-real-v126-lemma-fixed`.
Dataset có8file gốc +transportreceipt +2file sửa +corrected_assets_receipt.
Loader tiếp theo dùng bank_words_corrected.json; chưa cóCLIPtextembeddings.
Chỉ được kiểm local metadata/mock/static; chưa chạy API/notebook bởi agent.
